In [2]:
!pip install langchain-community chromadb pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/2

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from transformers import pipeline
from langgraph.graph import StateGraph
from typing import TypedDict

Step:-1 Upload PDF

In [4]:
from google.colab import files
uploaded = files.upload()

Saving Customer support Guide.pdf to Customer support Guide.pdf


Step:-2 Chunking

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 2


Step:- 3 Embeddings + Vector Database

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Create embedding model
embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Store in vector database
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding
)

print("Embeddings created and stored!")

/tmp/ipykernel_3574/2589430083.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created and stored!


Step:- 4 Retrieval (Get Relevant Chunks)

In [9]:
query = "What is the refund policy?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

docs = retriever.invoke(query)

print("Retrieved Chunks:\n")

for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

Retrieved Chunks:


--- Result 1 ---
Customer  Support  Guide   Refund  Policy:  Customers  can  request  a  refund  within  30  days  of  purchase.   Shipping  Policy:  Orders  are  delivered  within  5–7  business  days.   Cancellation

--- Result 2 ---
within  5–7  business  days.   Cancellation  Policy:  Orders  can  be  cancelled  before  shipment.   Contact:  Email:  support@example.com


Step:- 5 Answer Generation (RAG Output)

In [10]:
from transformers import pipeline

# Load model
generator = pipeline("text-generation", model="gpt2")

query = "What is the refund policy?"

# Retrieve context
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
docs = retriever.invoke(query)

context = "\n".join([doc.page_content for doc in docs])

# Prompt
prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question: {query}

Answer:
"""

# Generate answer
response = generator(prompt, max_new_tokens=40, do_sample=False)

answer = response[0]["generated_text"].split("Answer:")[-1].strip()

print("FINAL ANSWER:\n")
print(answer)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FINAL ANSWER:

Customer  Support  Guide   Refund  Policy:  Customers  can  request  a  refund  within  30  days  of  purchase.   Shipping  Policy:


Step 6: LangGraph Integration (Simple Version)

In [11]:
from langgraph.graph import StateGraph
from typing import TypedDict

# Define state
class GraphState(TypedDict):
    query: str
    answer: str

graph = StateGraph(GraphState)

# Node
def rag_node(state):
    query = state["query"]

    retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
Answer using the context.

Context:
{context}

Question: {query}
Answer:
"""

    response = generator(prompt, max_new_tokens=40, do_sample=False)
    answer = response[0]["generated_text"].split("Answer:")[-1].strip()

    return {"query": query, "answer": answer}

# Add node
graph.add_node("rag", rag_node)
graph.set_entry_point("rag")

# Compile + run
app = graph.compile()

result = app.invoke({"query": "What is the refund policy?"})

print("FINAL ANSWER:\n")
print(result["answer"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FINAL ANSWER:

Customer  Support  Guide   Refund  Policy:  Customers  can  request  a  refund  within  30  days  of  purchase.   Shipping  Policy:
